In [ ]:
!pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/raw_data
!unzip -q "/content/drive/MyDrive/dataset/FootballMatchDS.zip" -d "/content/raw_data"

In [ ]:
import os
import cv2
import glob
import shutil
import numpy as np
from sklearn.model_selection import train_test_split

base_raw_dir = "/content/raw_data"

images_dir = "/content/raw_data/Tagged_Images/Tagged Images"
masks_dir = "/content/raw_data/Masks/Masks"

output_yolo_dir = "/content/yolo_dataset"
os.makedirs(output_yolo_dir, exist_ok=True)


for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_yolo_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_yolo_dir, 'labels', split), exist_ok=True)

def mask_to_yolo_polygons(mask_path, class_id=0):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    H, W = mask.shape
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 10:
            cnt = cnt.flatten()
            poly = [str(class_id)]
            for i in range(0, len(cnt), 2):
                poly.append(f"{cnt[i] / W:.6f}")
                poly.append(f"{cnt[i+1] / H:.6f}")
            polygons.append(" ".join(poly))
    return polygons

image_files = []
for ext in ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg'):
    image_files.extend(glob.glob(os.path.join(images_dir, ext)))

image_files = sorted(image_files)
print(f"Trovate {len(image_files)} immagini nel dataset.")

train_files, temp_files = train_test_split(image_files, test_size=0.2, random_state=42)

val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

def process_dataset(file_list, split_name):
    contatore_successi = 0
    for img_path in file_list:
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0]

        mask_base_name = base_name.replace("frame", "mask")

        mask_path = None
        for ext in ('.png', '.PNG', '.jpg', '.jpeg'):
            temp_path = os.path.join(masks_dir, mask_base_name + ext)
            if os.path.exists(temp_path):
                mask_path = temp_path
                break

        if mask_path:
            dest_img = os.path.join(output_yolo_dir, 'images', split_name, filename)
            shutil.copy(img_path, dest_img)

            polygons = mask_to_yolo_polygons(mask_path, class_id=0)
            txt_filename = base_name + '.txt'
            dest_txt = os.path.join(output_yolo_dir, 'labels', split_name, txt_filename)

            with open(dest_txt, 'w') as f:
                for poly in polygons:
                    f.write(poly + '\n')
            contatore_successi += 1

    print(f"-> {split_name}: {contatore_successi} file elaborati con successo.")

print("\nElaborazione Train set (80%)...")
process_dataset(train_files, 'train')

print("Elaborazione Validation set (10%)...")
process_dataset(val_files, 'val')

print("Elaborazione Test set (10%)...")
process_dataset(test_files, 'test')

print("\nDataset YOLO creato con successo in /content/yolo_dataset!")

Trovate 1620 immagini nel dataset.

Elaborazione Train set (80%)...
-> train: 1296 file elaborati con successo.
Elaborazione Validation set (10%)...
-> val: 162 file elaborati con successo.
Elaborazione Test set (10%)...
-> test: 162 file elaborati con successo.

Dataset YOLO creato con successo in /content/yolo_dataset!


In [ ]:
yaml_content = f"""
train: /content/yolo_dataset/images/train
val: /content/yolo_dataset/images/val
test: /content/yolo_dataset/images/test

nc: 1
names: ['ad_panel']
"""

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("File data.yaml creato con successo per la classe 'ad_panel'!")

File data.yaml creato con successo per la classe 'ad_panel'!


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/YOLOv11_Seg_Results',
    name='ad_panel'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640

------------------------------------------------------------------------------------------------------------------

**Utilizzo del nuovo dataset**

In [ ]:
!mkdir -p /content/raw_data_v2

# Estrae il dataset zippato
!unzip -q "/content/drive/MyDrive/ProgettoDL/FootballAdvertisingBannersDetectionDS.zip" -d "/content/raw_data_v2"

print("Estrazione completata in /content/raw_data_v2!")

Estrazione completata in /content/raw_data_v2!


In [ ]:
import os
import cv2
import glob
import json
import shutil
import base64
import zlib
import numpy as np
from sklearn.model_selection import train_test_split

# Percorsi aggiornati
base_raw_dir = "/content/raw_data_v2"
images_dir = glob.glob(os.path.join(base_raw_dir, "**/images"), recursive=True)[0]
json_dir = glob.glob(os.path.join(base_raw_dir, "**/annotations"), recursive=True)[0]

output_yolo_dir = "/content/yolo_dataset_v2"
os.makedirs(output_yolo_dir, exist_ok=True)

# Crea le cartelle train/val/test
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_yolo_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_yolo_dir, 'labels', split), exist_ok=True)

def base64_2_mask(s):
    z = base64.b64decode(s)
    # Decomprime e converte in array numpy
    n = np.frombuffer(zlib.decompress(z), np.uint8)
    mask = cv2.imdecode(n, cv2.IMREAD_UNCHANGED)
    return mask

def json_to_yolo_polygons(json_path, class_id=0):
    with open(json_path, 'r') as f:
        data = json.load(f)

    img_h = data['size']['height']
    img_w = data['size']['width']
    polygons = []

    for obj in data.get('objects', []):
        if obj.get('geometryType') == 'bitmap':
            bitmap_data = obj['bitmap']['data']
            origin = obj['bitmap']['origin']
            mask = base64_2_mask(bitmap_data)

            if len(mask.shape) == 3:
                mask = mask[:, :, 0]

            h, w = mask.shape
            x, y = origin[0], origin[1]

            full_mask = np.zeros((img_h, img_w), dtype=np.uint8)

            y2 = min(y + h, img_h)
            x2 = min(x + w, img_w)

            full_mask[y:y2, x:x2] = mask[:y2-y, :x2-x]

            contours, _ = cv2.findContours(full_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                if cv2.contourArea(cnt) > 20:
                    cnt = cnt.flatten()
                    poly = [str(class_id)]
                    for i in range(0, len(cnt), 2):
                        poly.append(f"{cnt[i] / img_w:.6f}")
                        poly.append(f"{cnt[i+1] / img_h:.6f}")
                    polygons.append(" ".join(poly))
    return polygons

image_files = sorted(glob.glob(os.path.join(images_dir, "*.png")))
print(f"Trovate {len(image_files)} immagini.")

# Split 80/10/10
train_files, temp = train_test_split(image_files, test_size=0.2, random_state=42)
val_files, test_files = train_test_split(temp, test_size=0.5, random_state=42)

def process_batch(file_list, split_name):
    print(f"Elaborazione {split_name}...")
    for img_path in file_list:
        filename = os.path.basename(img_path)
        json_path = os.path.join(json_dir, filename + ".json")

        if os.path.exists(json_path):
            shutil.copy(img_path, os.path.join(output_yolo_dir, 'images', split_name, filename))
            polygons = json_to_yolo_polygons(json_path)
            txt_path = os.path.join(output_yolo_dir, 'labels', split_name, os.path.splitext(filename)[0] + ".txt")
            with open(txt_path, 'w') as f:
                f.write("\n".join(polygons))

process_batch(train_files, 'train')
process_batch(val_files, 'val')
process_batch(test_files, 'test')
print("Dataset pronto!")

Trovate 8851 immagini.
Elaborazione train...
Elaborazione val...
Elaborazione test...
Dataset pronto!


In [ ]:
yaml_content = f"""
train: /content/yolo_dataset_v2/images/train
val: /content/yolo_dataset_v2/images/val
test: /content/yolo_dataset_v2/images/test

nc: 1
names: ['ad_panel']
"""

with open('/content/yolo_dataset_v2/data.yaml', 'w') as f:
    f.write(yaml_content)

print("File data.yaml rigenerato con successo!")

File data.yaml rigenerato con successo!


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/YOLOv11_Seg_Results/ad_panel/weights/best.pt')

results = model.train(
    data='/content/yolo_dataset_v2/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/YOLOv11_Seg_Results',
    name='ad_panel_final_finetune',
    resume=False
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset_v2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/YOLOv11_Seg_Results/ad_panel/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ad_panel_final_finetune, nbs=64, nms=False, opset=N